In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().resolve()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

universe_500 = pd.read_csv(DATA_RAW / "universe_500.csv")["ticker"].astype(str).tolist()
monthly_returns_500 = pd.read_csv(DATA_PROCESSED / "monthly_returns_500.csv", parse_dates=["Date"], index_col=0)
momentum_panel_500 = pd.read_csv(DATA_PROCESSED / "momentum_panel_500.csv", parse_dates=["date"])

monthly_returns_500.index = pd.to_datetime(monthly_returns_500.index)
monthly_returns_500.columns = monthly_returns_500.columns.astype(str)

mom = momentum_panel_500.copy()
mom["ticker"] = mom["ticker"].astype(str)
mom["momentum_12m_z"] = mom.groupby("date")["momentum_12m"].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0) if s.std(ddof=0) not in [0, np.nan] else np.nan
)

ret = monthly_returns_500.copy()
ret_panel = ret.stack().reset_index()
ret_panel.columns = ["date", "ticker", "return"]

ret_panel["ticker"] = ret_panel["ticker"].astype(str)
ret_panel = ret_panel[ret_panel["ticker"].isin(universe_500)].copy()

price_only = mom[["date", "ticker", "momentum_12m", "momentum_12m_z"]].copy()
price_only = price_only[price_only["ticker"].isin(universe_500)].copy()

price_only.to_csv(OUT / "factor_B_price_only_500.csv", index=False)
ret_panel.to_csv(OUT / "factor_B_returns_500.csv", index=False)

coverage = pd.DataFrame({
    "model": ["B_price_only_500"],
    "n_tickers": [len(universe_500)],
    "n_return_rows": [len(ret_panel)],
    "n_momentum_rows": [len(price_only)],
    "start_date": [str(price_only["date"].min()) if len(price_only) else None],
    "end_date": [str(price_only["date"].max()) if len(price_only) else None]
})
coverage.to_csv(OUT / "factor_B_coverage.csv", index=False)

print(coverage)
print(price_only.head())
print(ret_panel.head())

              model  n_tickers  n_return_rows  n_momentum_rows  \
0  B_price_only_500        500          74000            65148   

            start_date             end_date  
0  2015-02-28 00:00:00  2026-05-31 00:00:00  
        date ticker  momentum_12m  momentum_12m_z
0 2015-02-28      A     -0.083319       -1.142891
1 2015-03-31      A      0.046393       -0.641806
2 2015-04-30      A      0.051219       -0.555722
3 2015-05-31      A      0.080462       -0.466389
4 2015-06-30      A      0.020971       -0.658370
        date ticker    return
0 2014-02-28   FDXF       NaN
1 2014-02-28   CIEN  0.053150
2 2014-02-28    ROP -0.011804
3 2014-02-28   VEEV  0.110412
4 2014-02-28   DELL       NaN
